In [1]:
import os

from toyaikit.mcp import MCPClient, SubprocessMCPTransport

In [2]:
DOCS_DIR = os.getcwd()

In [3]:
command = ["uv", "run", "python", "main.py"]
client = MCPClient(
    transport=SubprocessMCPTransport(
        server_command=command,
        workdir=DOCS_DIR,
    )
)

In [4]:
client.start_server()

Started server with command: uv run python main.py


In [5]:
client.initialize()

Sending initialize request...
Initialize response: {'protocolVersion': '2024-11-05', 'capabilities': {'logging': {}, 'prompts': {'listChanged': True}, 'resources': {'subscribe': False, 'listChanged': True}, 'tools': {'listChanged': True}}, 'serverInfo': {'name': 'Demo', 'version': '4.1.0'}}


{'protocolVersion': '2024-11-05',
 'capabilities': {'logging': {},
  'prompts': {'listChanged': True},
  'resources': {'subscribe': False, 'listChanged': True},
  'tools': {'listChanged': True}},
 'serverInfo': {'name': 'Demo', 'version': '4.1.0'}}

In [6]:
client.initialized()

Sending initialized notification...
Handshake completed successfully


In [7]:
client.get_tools()

Retrieving available tools...
Available tools: ['add_entry', 'get_file', 'search']


[{'_meta': {'fastmcp': {'tags': []}},
  'description': 'Add a new entry to the documentation database.',
  'inputSchema': {'properties': {'question': {'type': 'string',
     'description': 'The question to be added to the documentation database.'},
    'answer': {'type': 'string',
     'description': 'The corresponding answer to the question.'}},
   'required': ['question', 'answer'],
   'type': 'object',
   'additionalProperties': False},
  'name': 'add_entry',
  'title': 'Add Entry'},
 {'_meta': {'fastmcp': {'tags': []}},
  'description': "Retrieve a file's contents by filename.",
  'inputSchema': {'properties': {'filename': {'type': 'string',
     'description': 'The filename of the file to retrieve.'}},
   'required': ['filename'],
   'type': 'object',
   'additionalProperties': False},
  'name': 'get_file',
  'outputSchema': {'properties': {'result': {'type': 'string'}},
   'required': ['result'],
   'type': 'object',
   'x-fastmcp-wrap-result': True},
  'title': 'Get File'},
 {'_

In [8]:
result = client.call_tool("search", {"query": "AWS Lambda Python S3 video thumbnail generation 250x250"})

Calling tool 'search' with arguments: {'query': 'AWS Lambda Python S3 video thumbnail generation 250x250'}


In [9]:
result

{'_meta': {'fastmcp': {'wrap_result': True}},
 'content': [{'text': '[{"start":0,"content":"---\\ntitle: Build a Video Thumbnailer on AWS\\nh1: \\"Build a Video Thumbnailer with Pulumi using Lambdas, Containers, and Infrastructure on AWS\\"\\ndate: \\"2018-06-21\\"\\nmeta_desc: \\"Extracting a thumbnail from a video using a combination of Lambdas, containers, and connected data services and infrastructure.\\"\\nauthors: [\\"donna-malayeri\\"]\\ntags: [javascript, serverless, aws, containers]\\ncategory: tutorials\\n---\\n\\nPulumi makes it easy to build cloud applications that use a combination\\nof containers, lambdas, and connected data services and infrastructure:\\nColada apps.\\n\\nAn example of a Colada app is extracting a thumbnail from a video. A\\nserverless function can only run for 5 minutes, so we\'ll run a\\ncontainer in AWS Fargate to do the video processing.\\n\\nIn this app, a Lambda function is triggered whenever a new video is\\nuploaded to S3. This function launches 

In [10]:
from toyaikit.llm import AnthropicClient
from toyaikit.mcp import MCPTools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import AnthropicMessagesRunner

In [11]:
INSTRUCTIONS = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Pulumi, so you don't need to 
include the word 'pulumi' in search results
"""

In [12]:
mcp_tools = MCPTools(client)
chat_interface = IPythonChatInterface()

runner = AnthropicMessagesRunner(
    tools=mcp_tools,  # type: ignore
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,  # type: ignore
    llm_client=AnthropicClient(model="claude-opus-5"),
)


In [13]:
runner.run()

Retrieving available tools...
Available tools: ['add_entry', 'get_file', 'search']
-> Response received


Calling tool 'search' with arguments: {'query': 'S3 Lambda video thumbnail generation'}


-> Response received


Calling tool 'get_file' with arguments: {'filename': 'blog/aws-lambda-container-support/index.md'}


Calling tool 'search' with arguments: {'query': 'Python Lambda S3 bucket notification onObjectCreated event handler'}


-> Response received


Chat ended.


LoopResult(new_messages=[{'role': 'system', 'content': "\nYou're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nMake 3 iterations:\n\n1) in the first iteration, perform one search\n2) in the second interation, analyze the results from the previous search\n   and perform 2 more searches\n3) synthesise the results into the output\n\nIMPORTANT: at each step, give an explanation of why you want to perform \nsearch for this particular search query. It should be 2-3 sentences explaining\nthe logic of your decision.\n\nUse only facts from the knowledge base when answering.\nIf you cannot find the answer, inform the user.\n\nOur knowledge base is entirely about Pulumi, so you don't need to \ninclude the word 'pulumi' in search results\n"}, {'role': 'user', 'content': 'AWS Lambda Python S3 video thumbnail generation 250x250'}, {'role': 'assistant', 'content': [ThinkingBlock(signature='CAISrgQKwwIIEhgCKkATH/0x9yTfubHsibAO6isaeikl0z1L+0rh1/JqJVuQ